# DeepAgents 04 · 上下文治理、Rubric 评分与检索-卸载-委派

前三课把 deepagents 的「骨架」讲全了：`create_deep_agent`、七种后端、人工审核、记忆、子智能体、skills。
这一课补上官方文档里**剩下的三块高阶能力**，它们共同回答同一个问题：
**上下文是有限资源，怎么把它花在刀刃上？**

| 概念 | 是什么 | 本节的代码形态 |
|---|---|---|
| 上下文卸载 | 超大工具结果自动落盘，历史只留「路径 + 预览」 | `fetch_huge_document` → `result["files"]` |
| 文件权限 | `allow` / `deny` / `interrupt` 三档，先匹配先生效 | `FilesystemPermission(...)` |
| 任务规划 | v0.7 起 **opt-in** 的 `write_todos` | `TodoListMiddleware()` |
| Rubric 评分循环 | 运行时 LLM-as-a-judge，不达标就带着逐条反馈重做 | `RubricMiddleware(model=..., on_evaluation=...)` |
| 检索-卸载-委派 | 检索结果落盘，主线程只拿路径，子代理读文件回传结论 | `SubAgent(name="doc-reader")` |

> **本 notebook 由 `Agent/03_deepagents/` 下 3 个「官方补充」脚本合并而成**（源文件已归档到 `Agent/_py_source/`，**没有课案原版**）：
>
> | 源文件 | 角色 | 对应小节 |
> |---|---|---|
> | `14_上下文治理_官方补充.py` | 框架机制（**全离线**，用 ScriptedModel 驱动） | 第 1 节 |
> | `16_Rubric评分循环_官方补充.py` | Rubric 评分循环（**需模型**） | 第 2 节 |
> | `17_RAG_检索卸载委派_官方补充.py` | 检索-卸载-委派（**需模型 + embedding + rerank**） | 第 3 节 |

**官方文档**
- 上下文工程：<https://docs.langchain.com/oss/python/deepagents/context-engineering>
- 文件权限：<https://docs.langchain.com/oss/python/deepagents/permissions>
- Rubric 评分循环：<https://docs.langchain.com/oss/python/deepagents/rubric>
- 检索-卸载-委派：<https://docs.langchain.com/oss/python/deepagents/retrieval>

## 运行条件

| 项 | 说明 |
|---|---|
| 🟡 运行档位 | **需模型** —— 会真实调用根目录 `.env` 里配置的大模型 + 向量化 + 重排序 |
| 依赖 | `langchain` / `langgraph` / `langchain_openai` / `langchain_text_splitters` / `deepagents`（本项目 venv 已装） |
| 密钥 | `settings.api_key` / `settings.embedding.api_key` / `settings.rerank.api_key`（均已配置） |
| 前置服务 | 无 —— 不落库、不起服务；向量库用内存版 `InMemoryVectorStore` |
| 预计耗时 | 约 3~5 分钟（第 1 节离线秒跑；第 2/3 节各有数次真实模型调用） |

> ⚠️ 本课分「离线」与「需模型」两部分：
> **第 1 节（上下文治理）完全离线** —— 用 `ScriptedModel` 按剧本驱动，0 次真实模型调用，可复现；
> **第 2、3 节会真实调模型**，模型生成的中文措辞每次不同。
> 所以下面每个「预期输出」块里，标注「（措辞随模型变化）」的段落，你跑出来的文字大概率与这里不同 ——
> **要看的是结构**（判定结果、工具调用序列、字符数对照、落盘文件清单），不是那几句话本身。

## 本节地图

三个源文件讲的其实是同一条主线：**上下文有限 → 把大块内容挪出上下文 → 只保留「路径 + 结论」**。
其中最值得先看的是 Rubric 的生命周期（第 2 节会亲眼观察它）：

```mermaid
graph LR
    A["invoke(rubric=...)"] --> B["深度 agent 干活"]
    B --> C["评分器判定<br/>独立子代理"]
    C -->|satisfied| D["结束（达标）"]
    C -->|needs_revision| E{"还有迭代额度?"}
    E -->|有| F["注入逐条反馈 → 回炉重做"]
    F --> B
    E -->|无| G["max_iterations_reached"]
    C -->|failed| H["结束（rubric 没法评）"]
    C -->|grader_error| I["结束（评分器出错）"]
```

上面这张图等价于下面这张表（**裸 JupyterLab 不渲染 mermaid，看表即可**）：

| 终局判定 | 含义 | 本课在哪看 |
|---|---|---|
| `satisfied` | 达标，正常结束 | 第 2.2 节（换端点后可能看不到） |
| `needs_revision` | 不达标，但还有迭代额度 → 注入反馈重做 | 第 2.2 节 |
| `max_iterations_reached` | 不达标且额度用尽，保护性结束 | 第 2.4 节 |
| `failed` | 评分器认为 rubric 本身没法评 | 第 2.4 节 |
| `grader_error` | 评分器抛异常被兜住（**DeepSeek 上必现**） | 第 2.2 / 2.4 节 |

**与上下节的衔接**：上一课 `03_人工审核_记忆_子智能体_Skills.ipynb` 讲了「人来把关」，
这一课的第 1.4 节（`interrupt` 权限）和整个第 2 节（Rubric）讲的是**让框架/另一个模型来把关**；
下一课 `05_解释器PTC与异步子代理.ipynb` 会讲代码解释器与异步子代理。

## 0. 环境引导

notebook 的**工作目录默认是它自己所在的文件夹**，而本项目所有代码都写
`from config import settings`（`config.py` 在仓库根）。

所以每个 notebook 的第一格统一做一件事：**向上找到仓库根，切过去，并塞进 `sys.path`**。
少了这一格，后面每一格都会 `ModuleNotFoundError: config`。

In [ ]:
# ===== 环境引导（每个 notebook 的第一格，不要改）=====
import os
import sys
from pathlib import Path

NB_DIR = Path.cwd()                 # notebook 所在目录（chdir 之前先抓住）
ROOT = NB_DIR
while not (ROOT / "config.py").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("没找到 config.py：请在 Python_Base 仓库内运行本 notebook")
    ROOT = ROOT.parent

os.chdir(ROOT)                      # 让相对路径（data/、output.txt 等）都相对仓库根
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

WORKDIR = NB_DIR / "tmp_nb_work"    # 本 notebook 的临时工作目录（已被 .gitignore 覆盖）
WORKDIR.mkdir(exist_ok=True)

print("仓库根：", ROOT)
print("临时目录：", WORKDIR)

### 预期输出

%%%OUTPUT:1%%%

第二格做**前置条件自检**：依赖装没装、三组密钥（模型 / 向量化 / 重排序）齐不齐。

本课第 1 节离线（不碰模型），但第 2、3 节要调模型、向量化和重排序，
所以自检比「纯模型」课多查两组：`settings.embedding.*` 与 `settings.rerank.*`。
缺了不会立刻崩，而是打印中文提示 —— 这样你在 JupyterLab 里点「Run All」时，
看到的是明确的「缺什么」，而不是一屏 `ModuleNotFoundError` / `AuthenticationError` 的堆栈。

In [ ]:
# ===== 前置条件自检：缺什么就打印中文提示，缺密钥时把 READY 置 False =====
import importlib.util

READY = True

for pkg in ("langchain", "langchain_core", "langchain_openai", "langchain_text_splitters", "deepagents"):
    if importlib.util.find_spec(pkg) is None:
        print(f"[缺依赖] 未安装 {pkg}，请先在本项目 venv 里装好再运行本 notebook")
        READY = False

try:
    from config import settings
except Exception as exc:          # noqa: BLE001 —— 自检格不该把整个 notebook 带崩
    print(f"[缺配置] 读不到 config.settings：{type(exc).__name__}: {exc}")
    settings = None
    READY = False

if settings is not None:
    missing = [
        name for name in ("api_key", "base_url", "model_name")
        if not getattr(settings, name, "")
    ]
    if missing:
        print("[缺密钥] 根目录 .env 里这几项还是空的：", ", ".join(missing))
        READY = False
    emb_missing = [
        "embedding.api_key" if not settings.embedding.api_key else "",
        "embedding.model" if not settings.embedding.model else "",
        "rerank.api_key" if not settings.rerank.api_key else "",
        "rerank.model" if not settings.rerank.model else "",
    ]
    emb_missing = [m for m in emb_missing if m]
    if emb_missing:
        print("[缺密钥] 根目录 .env 里这几项还是空的：", ", ".join(emb_missing))
        print("        第 3 节的向量化/重排序会失败，请先在 .env 里补齐")
        READY = False
    if not missing and not emb_missing:
        print(f"✔ 依赖齐全，模型配置已就绪：{settings.model_name} @ {settings.base_url}")
        print(f"✔ 向量化 {settings.embedding.model} / 重排序 {settings.rerank.model} 已配置")

print("前置条件自检：", "通过" if READY else "未通过")

### 预期输出

%%%OUTPUT:2%%%

## 1. 上下文治理：框架机制（离线，ScriptedModel 驱动）

对应源文件 `14_上下文治理_官方补充.py`。deepagents 与普通 agent 最本质的差别之一：
**它自带上下文压缩**，不用你加中间件。官方 `context-engineering.mdx` 讲了三层机制：

1. **卸载（offloading）**：工具结果过大时，把完整内容写进 backend 文件，
   历史里只留「文件路径 + 少量预览」；
2. **摘要（summarization）**：会话接近模型窗口上限时，把旧对话压成结构化摘要；
3. **按需 compact**：可选地给代理一个 `compact_conversation` 工具，自己触发压缩。

本节演示第 1 层（最容易观察）+ 两样配套的「治理」机制（任务规划 `write_todos`、文件权限 `FilesystemPermission`）。

为什么全离线：这四件事讲的都是**框架机制**（工具在不在、权限拦不拦、结果卸不卸载），
用 `ScriptedModel`（继承 `ChatOpenAI`、覆写 `_generate` 按剧本返回消息）即可 100% 复现，
真模型只会引入随机性。

### 1.1 剧本模型 + 工具消息打印

`ScriptedModel` 是离线可复现的关键：它只覆写 `_generate`（真正发 HTTP 请求那一步），
`bind_tools` / 消息校验等框架方法沿用真实现 —— 所以断网也能跑。
`make_scripted` 里的 `api_key` / `base_url` 传假值即可，永远不会被真正用到。

In [ ]:
from langchain.agents.middleware import TodoListMiddleware
from langchain_core.messages import AIMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command
from pydantic import PrivateAttr

from deepagents import FilesystemPermission, create_deep_agent


def ai_tool_call(name: str, args: dict, call_id: str) -> AIMessage:
    """构造一条「模型要调工具」的 AIMessage。"""
    return AIMessage(
        content="",
        tool_calls=[{"name": name, "args": args, "id": call_id, "type": "tool_call"}],
    )


class ScriptedModel(ChatOpenAI):
    """按剧本依次吐消息的假模型（与 02_langchain/11_内置中间件_官方补充.py 同名类同一手法）。

    继承 ChatOpenAI、只覆写 _generate —— bind_tools / 消息校验等框架方法沿用真实现，
    唯一被替换的是「真正发 HTTP 请求」那一步，所以断网也能跑。
    """

    _script: list = PrivateAttr(default_factory=list)
    _cursor: int = PrivateAttr(default=0)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs):
        message = self._script[self._cursor]
        self._cursor += 1
        return ChatResult(generations=[ChatGeneration(message=message)])


def make_scripted(script: list) -> ScriptedModel:
    """造一个剧本模型。api_key / base_url 传假值即可：永远不会被真正用到。"""
    model = ScriptedModel(model="scripted", api_key="offline", base_url="http://localhost:9")
    model._script = script
    return model


def show_tool_messages(result: dict, limit: int = 90) -> None:
    """打印结果里的工具消息（工具到底干了什么，全在这里）。"""
    for message in result.get("messages", []):
        if message.type == "tool":
            name = getattr(message, "name", "?")
            body = str(message.content).replace("\n", " ")
            print(f"    [ToolMessage] {name}: {body[:limit]}")

### 1.2 Demo 1：`write_todos` 是 opt-in —— 不传 `TodoListMiddleware` 就没有

老版本 deepagents 默认就给代理装 `write_todos`（任务规划工具）；**0.7 起改为显式选择**。
这个变更不讲清楚，升级后会出现「代理突然不会做计划了」的困惑 ——
本 Demo 用**对照实验**把差异摆出来：

- A：默认 `create_deep_agent`（不传 middleware）→ 调 `write_todos` 要么没有、要么直接失败；
- B：显式传 `middleware=[TodoListMiddleware()]` → `todos` 被正确写入 state。

> 注意 `TodoListMiddleware` 来自 **langchain**（`from langchain.agents.middleware import ...`），
> 不是 deepagents 自己导出的。

In [ ]:
def demo_1_todo_opt_in() -> None:
    print("=" * 70)
    print("Demo 1：write_todos 是 opt-in —— 不传 TodoListMiddleware 就没有这个工具")
    print("=" * 70)

    # ---- A. 默认的 create_deep_agent：没有 write_todos ----
    default_agent = create_deep_agent(
        model=make_scripted([
            ai_tool_call("write_todos", {"todos": [{"content": "调研", "status": "pending"}]}, "c1"),
            AIMessage(content="（默认代理这一轮不会有待办清单）"),
        ]),
    )
    print("\nA. 默认 create_deep_agent（不传 middleware）")
    try:
        result = default_agent.invoke({"messages": [{"role": "user", "content": "帮我规划三件事"}]})
        show_tool_messages(result)
        todos = result.get("todos")
        print(f"    state 里的 todos：{todos!r}")
        todos = result.get("todos")
        if not todos:
            print("    ↑ 默认栈里没有 TodoListMiddleware，write_todos 不是可用工具")
        else:
            print(f"    ↑ 本次默认栈**居然带了** TodoList（todos={todos}）—— "
                  "说明这个版本的默认栈变了，结论要以运行结果为准")
    except Exception as exc:  # noqa: BLE001
        print(f"    调用 write_todos 直接失败：{type(exc).__name__}: {str(exc)[:80]}")
        print("    ↑ 同样证明：默认代理没有这个工具")

    # ---- B. 显式挂上 TodoListMiddleware：立刻可用 ----
    todo_agent = create_deep_agent(
        model=make_scripted([
            ai_tool_call(
                "write_todos",
                {"todos": [
                    {"content": "调研竞品", "status": "in_progress"},
                    {"content": "设计接口", "status": "pending"},
                    {"content": "编写测试", "status": "pending"},
                ]},
                "c1",
            ),
            AIMessage(content="已排好待办清单。"),
        ]),
        # 注意：TodoListMiddleware 来自 **langchain**（deepagents 自己不导出它）
        middleware=[TodoListMiddleware()],
    )
    print("\nB. create_deep_agent(middleware=[TodoListMiddleware()])")
    result = todo_agent.invoke({"messages": [{"role": "user", "content": "帮我规划三件事"}]})
    show_tool_messages(result)
    todos = result.get("todos") or []
    print(f"    state 里的 todos（{len(todos)} 条）：")
    for index, todo in enumerate(todos, start=1):
        print(f"      {index}. [{todo.get('status')}] {todo.get('content')}")
    assert todos, "挂了 TodoListMiddleware 之后 todos 应该有内容"
    print(
        "    ↑ 同一句用户输入，差别只在于**有没有把 TodoListMiddleware 传进 middleware** ——\n"
        "      这就是 v0.7 的 opt-in 变更：想要规划能力，得自己声明。"
    )


demo_1_todo_opt_in()

### 预期输出

%%%OUTPUT:3%%%

### 1.3 Demo 2：`FilesystemPermission(mode='deny')` —— 越权写入被拒

权限规则形状（官方 `permissions.mdx`）：

```text
FilesystemPermission(operations=["read"|"write"], paths=["glob 模式"],
                     mode="allow" | "deny" | "interrupt")
```

- `operations`：`write` 覆盖 `write_file` / `edit_file` / `delete`；`read` 覆盖 `ls` / `read_file` / `glob` / `grep`；
- 求值顺序：**先匹配先生效**；一条都不匹配时默认放行（宽松默认）；
- `paths` 建议锚定（`"/secrets/**"`），避免 `"/**/secrets"` 这种到处误伤的写法。

本 Demo 的关键结论：**拒绝的表现是工具返回一句说明，不是抛异常**（模型照常继续执行）。

In [ ]:
def demo_2_permission_deny() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：FilesystemPermission(mode='deny') —— 越权写入被拒")
    print("=" * 70)

    agent = create_deep_agent(
        model=make_scripted([
            # 剧本：先试图往禁区写，再往允许的目录写
            ai_tool_call("write_file", {"file_path": "/secrets/token.txt", "content": "机密"}, "c1"),
            ai_tool_call("write_file", {"file_path": "/work/notes.txt", "content": "普通笔记"}, "c2"),
            AIMessage(content="该写的写了，该拦的拦了。"),
        ]),
        permissions=[
            # 禁区：写 /secrets/** 一律拒绝
            FilesystemPermission(operations=["write"], paths=["/secrets/**"], mode="deny"),
            # 白名单：/work/** 明确允许（其实默认就放行，这里写出来是为了让规则自解释）
            FilesystemPermission(operations=["read", "write"], paths=["/work/**"], mode="allow"),
        ],
    )
    result = agent.invoke({"messages": [{"role": "user", "content": "写两个文件"}]})
    show_tool_messages(result, limit=120)
    # 结论按**实际拿到的工具消息**判断打印，不写死 —— 权限规则一变，这里要能如实反映
    tool_texts = [str(m.content) for m in result["messages"] if m.type == "tool"]
    denied = [text for text in tool_texts if "permission denied" in text.lower() or "拒绝" in text]
    written = [text for text in tool_texts if "Updated file" in text]
    print(
        f"    ↑ 实测：{len(denied)} 条被权限拒绝、{len(written)} 条写入成功 ——\n"
        "      拒绝的表现是**工具返回一句说明**（不是抛异常）；规则「先匹配先生效」，没匹配上的默认放行。\n"
        "      注意路径是 StateBackend 的虚拟文件系统路径，都以 / 开头。"
    )


demo_2_permission_deny()

### 预期输出

%%%OUTPUT:4%%%

### 1.4 Demo 3：`FilesystemPermission(mode='interrupt')` —— 越权写入转人工审批

这是本节最值得学的一招：把「权限」和「人工审核」缝在一起 ——
命中规则的写操作**不会被执行**，而是抛出一个**人工中断**，等审批决定：

```text
Command(resume={"decisions": [{"type": "approve"}]})   → 批准 → 真的执行
Command(resume={"decisions": [{"type": "reject"}]})    → 拒绝 → 不执行
```

它要求 checkpointer（要能暂停/恢复），且恢复格式与工具中断完全一致。
本 Demo 分两条路径：A 批准（工具真正执行）、B 拒绝（文件不写入）。

In [ ]:
def demo_3_permission_interrupt() -> None:
    print("\n" + "=" * 70)
    print("Demo 3：FilesystemPermission(mode='interrupt') —— 越权写入转人工审批")
    print("=" * 70)

    def build_agent(script: list):
        return create_deep_agent(
            model=make_scripted(script),
            permissions=[
                FilesystemPermission(operations=["write"], paths=["/protected/**"], mode="interrupt"),
            ],
            # interrupt 模式必须配 checkpointer，否则没法暂停/恢复
            checkpointer=MemorySaver(),
        )

    # ---- A. 批准路径 ----
    agent = build_agent([
        ai_tool_call("write_file", {"file_path": "/protected/report.txt", "content": "报告"}, "c1"),
        AIMessage(content="已按审批结果处理。"),
    ])
    config = {"configurable": {"thread_id": "perm-approve"}}
    first = agent.invoke({"messages": [{"role": "user", "content": "写一份报告到受保护目录"}]}, config)
    print("\nA. 批准路径")
    if "__interrupt__" in first:
        payload = first["__interrupt__"][0].value
        print(f"    第一次 invoke 返回中断（工具还没执行）：{str(payload)[:110]}")
    else:
        print(f"    第一次 invoke 没有中断：{str(first.get('messages', [])[-1].content)[:80]}")
    second = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config)
    show_tool_messages(second, limit=110)
    print("    ↑ 批准后工具才真正执行（ToolMessage 是成功写入的结果）")

    # ---- B. 拒绝路径 ----
    agent = build_agent([
        ai_tool_call("write_file", {"file_path": "/protected/report.txt", "content": "报告"}, "c1"),
        AIMessage(content="已按审批结果处理。"),
    ])
    config = {"configurable": {"thread_id": "perm-reject"}}
    agent.invoke({"messages": [{"role": "user", "content": "再写一份到受保护目录"}]}, config)
    rejected = agent.invoke(Command(resume={"decisions": [{"type": "reject"}]}), config)
    print("\nB. 拒绝路径")
    show_tool_messages(rejected, limit=110)
    rejected_msgs = [str(m.content) for m in rejected.get("messages", []) if getattr(m, "type", "") == "tool"]
    print(f"    （本次工具消息：{rejected_msgs[-1][:70] if rejected_msgs else '（无）'}）")
    print("    ↑ 拒绝后文件没有被写入 —— 权限 + 人工审核的组合，比单纯的 deny 更适合\n"
          "      「本来该允许、但要有人签字」的场景")


demo_3_permission_interrupt()

### 预期输出

%%%OUTPUT:5%%%

### 1.5 Demo 4：内置上下文压缩 —— 超长工具结果被「卸载」到文件系统

造一个返回 10 万字符的工具，看历史被替换成什么。
卸载后的完整内容去哪了？StateBackend 把它写进了 state 的虚拟文件系统（`result["files"]`），
可被 `read_file` / `ls` 读回 —— 这正是「上下文工程」的核心手法：
**把大块内容从上下文挪到文件系统，上下文窗口只花在「路径 + 预览」上。**
课案 03~09 讲的七种后端，正是这套机制的存放载体。

In [ ]:
BIG_TEXT_LEN = 100_000


@tool
def fetch_huge_document(topic: str) -> str:
    """返回一份超长文档（用于触发上下文卸载）。"""
    # 造 10 万字符：真实场景是爬下来的网页 / 大日志 / 长检索结果
    return f"# {topic}\n" + ("这是一行重复的正文内容。" * (BIG_TEXT_LEN // 12))


def demo_4_context_offloading() -> None:
    print("\n" + "=" * 70)
    print("Demo 4：内置上下文压缩 —— 超长工具结果被卸载到文件系统")
    print("=" * 70)

    agent = create_deep_agent(
        model=make_scripted([
            ai_tool_call("fetch_huge_document", {"topic": "长文档"}, "c1"),
            AIMessage(content="文档已读，我按预览内容回答。"),
        ]),
        tools=[fetch_huge_document],
    )
    result = agent.invoke({"messages": [{"role": "user", "content": "读一下那份长文档"}]})

    tool_messages = [m for m in result["messages"] if m.type == "tool"]
    if not tool_messages:
        print("  本轮没有工具消息（不符合预期）")
        return
    history_len = len(str(tool_messages[0].content))
    print(f"  工具原始返回：约 {BIG_TEXT_LEN:,} 字符")
    print(f"  历史里留下的工具消息：{history_len:,} 字符")
    print(f"  历史内容预览：{str(tool_messages[0].content)[:150]!r}")

    # 卸载后的完整内容去哪儿了？StateBackend 把它写进了 state 的虚拟文件系统
    files = result.get("files")
    if isinstance(files, dict) and files:
        print(f"  state 里的虚拟文件系统：{sorted(files.keys())}")
        for path, payload in files.items():
            content = payload.get("content", payload) if isinstance(payload, dict) else payload
            print(f"    {path}：{len(str(content)):,} 字符（完整内容在这里，可被 read_file/ls 读回）")
        print(
            "  ↑ 这就是「上下文工程」的核心手法：**把大块内容从上下文挪到文件系统**，\n"
            "    模型需要细节时再用 read_file/grep 取 —— 上下文窗口只花在「路径 + 预览」上。\n"
            "    课案 03~09 讲的七种后端，正是这套机制的存放载体。"
        )
    else:
        print("  （state 里没有 files 字段 —— 该版本可能把卸载内容放在别处，请看上面的历史预览）")


demo_4_context_offloading()
print("\n全部 Demo 执行完毕（0 次真实模型调用，离线可复现）。")

### 预期输出

%%%OUTPUT:6%%%

## 2. Rubric 评分循环：让 agent 自我验收（需模型）

对应源文件 `16_Rubric评分循环_官方补充.py`。官方 `rubric.mdx` 的定位原话：

> 「Some agent tasks have a clear definition of "done" that the working model alone
> cannot reliably hit on the first try... `RubricMiddleware` lets you declare
> *what done looks like* as a rubric and have the agent **self-evaluate and iterate**
> until the rubric is satisfied, or until a configured maximum iteration cap is hit.」

它是 **LLM-as-a-judge 的运行时版本**：LangSmith 的同类做法是离线批量评分；
`RubricMiddleware` 把它搬到运行时 —— agent 出结果后，一个**独立的评分器子代理**拿 rubric
审这份结果、给逐条判定，不满意就把**逐条反馈**注入对话让 agent 重做。

> ⚠️ **两个必须知道的版本/行为要点（本地实测 + 官方说明）**：
>
> A. `RubricMiddleware` 需要 **deepagents>=0.6.5**，且官方标注 **beta**（API 可能变）；
> B. **不传 rubric 时它完全是 no-op**（`before_agent` / `after_agent` 直接返回），
>    所以可以无条件挂在中间件栈里（第 2.3 节用零额外模型调用验证了这一点）。

### 2.1 准备：模型 + `on_evaluation` 回调

评分器与被评的 agent 都用 `.env` 里的模型（生产里通常给评分器配更便宜的模型）。
`on_evaluation` 回调是观察评分过程的主要手段：每次评分后触发。

`RubricEvaluation` 字段（实测）：`grading_run_id` / `iteration` / `result` /
`explanation` / `criteria` / `unverified`。其中 `criteria` 实测是 **list[dict]**，
每个元素形如 `{'name': '标准描述', 'passed': True/False}` —— 直接按 dict 读即可。

In [ ]:
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph.checkpoint.memory import InMemorySaver

from deepagents import RubricMiddleware, create_deep_agent
from deepagents.middleware.rubric import RubricEvaluation

from config import settings

# 评分器与被评的 agent 都用 .env 里的模型（生产里通常给评分器配更便宜的模型）
model = init_chat_model(
    model_provider="openai",
    model=settings.model_name,
    api_key=settings.api_key,
    base_url=settings.base_url,
    max_retries=0,
    # 评分循环一次要跑 6+ 次模型调用（主 agent + 评分器，每轮两次）；
    # 本机网关在负载高时单次请求可能超过两分钟，给足 5 分钟避免整跑崩在超时上。
    # （换成 DeepSeek 端点后单次基本 2~10 秒返回，这个值只当兜底上限用。）
    timeout=300,
)

# 每次评分回调都记下来，运行结束统一打印（也可直接 print）
evaluations: list[RubricEvaluation] = []


def log_evaluation(ev: RubricEvaluation) -> None:
    """官方 on_evaluation 回调：每次评分后触发，是观察评分过程的主要手段。

    RubricEvaluation 字段（实测）：grading_run_id / iteration / result /
    explanation / criteria / unverified。

    criteria 实测是 **list[dict]**，每个元素形如 `{'name': '标准描述', 'passed': True/False}`
    （类型注解上它是 CriterionPass | CriterionFail 的判别联合，经 RubricEvaluation
    取出来时已是序列化后的 dict）—— 直接按 dict 读即可。
    """
    evaluations.append(ev)
    criteria = ev.get("criteria") or []
    print(f"    [评分回调] 第 {ev.get('iteration')} 轮 → 判定：{ev.get('result')}")
    print(f"               说明：{str(ev.get('explanation'))[:90]}")
    for index, item in enumerate(criteria, start=1):
        if isinstance(item, dict):
            mark = "✔" if item.get("passed") else "✘"
            print(f"               标准 {index}：{mark} {str(item.get('name'))[:60]}")
        else:   # 兜底：万一某版本给的是对象
            detail = getattr(item, "criterion", None) or getattr(item, "name", None)
            print(f"               标准 {index}（{type(item).__name__}）：{str(detail or item)[:70]}")

### 2.2 Demo 1：基础评分循环 —— 不达标就带着逐条反馈重做

任务故意选「有明确硬指标」的：一段介绍必须覆盖 3 个指定要点、且不超过指定字数。
这种任务第一次就完全达标的概率不高，正好用来观察「判定 → 注入反馈 → 重做」的循环。

官方用法：rubric 通过**调用时的 state** 传入（`invoke({..., "rubric": RUBRIC})`），
且必须配 checkpointer（循环靠它恢复状态）。

> ⚠️ **DeepSeek 端点上的预期结果（本机实测）**：
> 评分器依赖**结构化输出** —— deepagents 内部用 `ToolStrategy` 让评分器吐结构化逐条判定，
> 而 `ToolStrategy` 固定发**强制 `tool_choice`**，思考模型端点（本机 `deepseek-flash`）跑不了，
> 会返回 `400 Thinking mode does not support this tool_choice`。
> 中间件把评分器异常**兜成 `grader_error`**（不是抛穿），所以脚本仍正常退出，
> 但你**看不到「判定 → 反馈 → 重做」的循环**，`on_evaluation` 的 `explanation` 是
> 「Grader raised OpenAIInvalidRequestError (HTTP 400)」。
> **换端点后先看 on_evaluation 判定是不是 grader_error，别误判成代码有问题。**

In [ ]:
RUBRIC = (
    "这份介绍必须同时满足以下三条，缺一不可：\n"
    "1. 提到「文件系统」这一能力；\n"
    "2. 提到「子代理」这一能力；\n"
    "3. 全文不超过 120 个汉字。"
)


def demo_1_basic_rubric() -> None:
    print("=" * 70)
    print("Demo 1：基础评分循环 —— 不达标就带着逐条反馈重做")
    print("=" * 70)

    evaluations.clear()
    agent = create_deep_agent(
        model=model,
        middleware=[
            # model 是**关键字参数**（实测签名：只有 model 必填，其余都有默认值）
            RubricMiddleware(model=model, max_iterations=3, on_evaluation=log_evaluation),
        ],
        checkpointer=InMemorySaver(),   # 评分循环要求可恢复，必须配 checkpointer
    )
    config = {"configurable": {"thread_id": "rubric-basic"}}
    result = agent.invoke(
        {
            "messages": [{"role": "user", "content": "写一段 DeepAgents 的能力介绍。"}],
            "rubric": RUBRIC,          # ← 官方用法：rubric 通过**调用时的 state** 传入
        },
        config,
    )
    print(f"\n  最终输出：{str(result['messages'][-1].content)[:200]}")
    print(f"  评分轮次：{len(evaluations)}，最终判定："
          f"{evaluations[-1].get('result') if evaluations else '（没有评分记录）'}")
    print(
        "  ↑ 评分器是**独立子代理**（不是让主模型自评）—— 它拿 rubric 逐条判、给逐条反馈，\n"
        "    反馈被注入对话后主 agent 重做。生产里给评分器配更便宜的模型即可控成本。"
    )


demo_1_basic_rubric()

### 预期输出

%%%OUTPUT:7%%%

### 2.3 Demo 2：不传 rubric → 中间件不介入（可无条件常驻）

官方 note：「The middleware activates only when a caller passes a `rubric` on
invocation state. With no rubric, both before_agent and after_agent return without
modifying state, so the middleware is safe to include unconditionally.」

这个性质很重要：**可以把它常驻在中间件栈里**，只在需要严格验收的调用上传 rubric。
所以「质检流程」不必另建一个 agent：同一个 agent，普通调用走直答，需要验收的调用带 rubric。

In [ ]:
def demo_2_noop_without_rubric() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：不传 rubric → 中间件不介入（可无条件常驻）")
    print("=" * 70)

    evaluations.clear()
    agent = create_deep_agent(
        model=model,
        middleware=[RubricMiddleware(model=model, max_iterations=3, on_evaluation=log_evaluation)],
        checkpointer=InMemorySaver(),
    )
    config = {"configurable": {"thread_id": "rubric-noop"}}
    result = agent.invoke(
        {"messages": [{"role": "user", "content": "用一句话说明什么是深度智能体。"}]},
        config,
    )
    print(f"  输出：{str(result['messages'][-1].content)[:120]}")
    print(f"  评分回调次数：{len(evaluations)} ← 0 表示中间件没有介入（no-op 成立）")
    print(
        "  ↑ 所以「质检流程」不必另建一个 agent：同一个 agent，\n"
        "    普通调用走直答，需要严格验收的调用带 rubric —— 这也是官方推荐的无条件挂载方式。"
    )


demo_2_noop_without_rubric()

### 预期输出

%%%OUTPUT:8%%%

### 2.4 Demo 3：评分器带工具取证 + 刻意不达标 → 迭代上限保护

官方参数表里 `tools` 是「评分器可以调用来收集证据的工具（跑测试、数字数、读文件）」。
本 Demo 给评分器一个「统计汉字数」的工具，让字数这条标准有**客观依据**而不是靠模型目测；
rubric 故意设成做不到的样子，用来观察迭代上限的保护（避免无限循环）。

两个要点：
1. `tools=` 让评分器**取证**，判定比纯目测可信；
2. 非 satisfied 终止时**消息不被改写** —— 想知道结局就看 `on_evaluation` 回调
   （或 v3 流的 `rubric_evaluation_end` 事件），别去猜最后一条消息的含义。

In [ ]:
@tool
def count_chinese_chars(text: str) -> str:
    """统计一段文字里的汉字个数（供评分器取证）。"""
    count = sum(1 for ch in text if "\u4e00" <= ch <= "\u9fff")
    return f"汉字数量：{count}"


IMPOSSIBLE_RUBRIC = (
    "必须同时满足：\n"
    "1. 全文正好 7 个汉字（不多不少）；\n"
    "2. 同时完整介绍文件系统、子代理、任务规划三项能力；\n"
    "3. 不出现任何标点符号。"
)


def demo_3_grader_tools_and_cap() -> None:
    print("\n" + "=" * 70)
    print("Demo 3：评分器带工具取证 + 刻意不达标 → 迭代上限保护")
    print("=" * 70)

    evaluations.clear()
    agent = create_deep_agent(
        model=model,
        middleware=[
            RubricMiddleware(
                model=model,
                tools=[count_chinese_chars],   # 评分器可用它取证（数汉字）
                max_iterations=2,              # 故意给小值，快点撞上限
                on_evaluation=log_evaluation,
            ),
        ],
        checkpointer=InMemorySaver(),
    )
    config = {"configurable": {"thread_id": "rubric-cap"}}
    result = agent.invoke(
        {
            "messages": [{"role": "user", "content": "介绍一下 DeepAgents。"}],
            "rubric": IMPOSSIBLE_RUBRIC,
        },
        config,
    )
    final_verdict = evaluations[-1].get("result") if evaluations else "（没有评分记录）"
    print(f"\n  评分轮次：{len(evaluations)}，最终判定：{final_verdict}")
    print(f"  最后一条消息（注意：非 satisfied 终止时中间件**不改写**消息）："
          f"{str(result['messages'][-1].content)[:120]}")
    if final_verdict == "max_iterations_reached":
        print("  ✔ 命中迭代上限保护：评分器还想改，但额度用完了，运行正常结束（不会死循环）")
    print(
        "  ↑ 两个要点：\n"
        "    ① `tools=` 让评分器**取证**（数汉字、跑测试、读文件），判定比纯目测可信；\n"
        "    ② 非 satisfied 终止时消息不被改写 —— 想知道结局就看 on_evaluation 回调\n"
        "       （或 v3 流的 rubric_evaluation_end 事件），别去猜最后一条消息的含义。"
    )


demo_3_grader_tools_and_cap()
print("\n全部 Demo 执行完毕。")

### 预期输出

%%%OUTPUT:9%%%

## 3. RAG 检索-卸载-委派（需模型 + embedding + rerank）

对应源文件 `17_RAG_检索卸载委派_官方补充.py`。官方 `retrieval.mdx` 把 RAG 分成三种架构：

| 架构 | 描述 | 控制力 | 灵活性 | 延迟 |
|---|---|---|---|---|
| 2-Step RAG | 检索永远发生在生成之前，简单可预测 | 高 | 低 | 快 |
| Agentic RAG | agent 自己决定**何时、怎么**检索 | 低 | 高 | 不定 |
| Hybrid | 两者结合 + 校验环节 | 中 | 中 | 不定 |

以及 DeepAgents 特有的四种落地模式（`rag.mdx`），本节实现的是**第四个（检索-卸载-委派）**：

| 模式 | 本仓库对应 |
|---|---|
| ① 技能引导检索 | `02_langchain/17_Skills渐进披露_官方补充.py` |
| ② Rubric 校验接地 | 本节第 2 节（Rubric 评分循环） |
| ③ Todo 驱动调查 | 本节第 1.2 节（write_todos） |
| ④ **检索-卸载-委派** | **本节（唯一还没做的那个）** |

④ 为什么值得单独讲：普通 RAG 把检索全文**塞进主线程上下文**，文档一多就撑爆窗口。
官方的做法是：**把 chunk 卸载到文件系统**，主 agent 只拿到「文件路径 + 极短预览」，
再派子代理去读、去搜、去总结 —— 主线程只承载结论，全文留在文件里。
本节把这个差异**量化**出来（Demo 2 vs Demo 3 打印两种做法的上下文字符数）。

### 3.1 准备：模型 + embedding + 知识库 + 重排序

本节的向量化 / 重排序模型来自 `.env`：`BAAI/bge-m3`（1024 维）与 `BAAI/bge-reranker-v2-m3`，
都是 SiliconFlow 端点。`SiliconFlowReranker` 走原生 `/rerank` 接口（`urllib` 直连，绕开代理）。
知识库素材故意写得长一些，模拟「文档页」，让上下文的差距看得出来。

In [ ]:
import json
import tempfile
import urllib.request
from pathlib import Path

from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

from deepagents import SubAgent, create_deep_agent
from deepagents.backends import FilesystemBackend

from config import settings

llm = init_chat_model(
    model_provider="openai",
    model=settings.model_name,          # grok-4.6
    api_key=settings.api_key,
    base_url=settings.base_url,
    max_retries=0,
    # 显式给足超时：本机网关在负载高（或长输出/高推理量）时单次请求可能超过两分钟。
    # 实测：120 秒会在网关繁忙时抛 OpenAITimeoutError；放宽到 300 秒后稳定通过。
    timeout=300,
)

embeddings = OpenAIEmbeddings(
    model=settings.embedding.model,     # BAAI/bge-m3
    api_key=settings.embedding.api_key,
    base_url=settings.embedding.base_url,
    check_embedding_ctx_length=False,
    # 显式超时：端点慢时快速失败，而不是无限挂住（实测踩过：没设超时时整跑会卡死）
    request_timeout=60,
    max_retries=1,
)

# ================================================================
# 知识库素材：故意写得长一些，模拟"文档页"，让上下文差距看得出来
# ================================================================
DOCS = {
    "langgraph-persistence.md": """# LangGraph 持久化详解

LangGraph 的持久化由 checkpointer 承担。每执行完一个 super-step，图的状态快照就会被
写入 checkpointer；thread_id 是"这条会话线"的唯一标识，所有快照都挂在它下面。

中断恢复的原理：当节点里调用 interrupt() 时，框架抛出一个特殊信号，把当前状态落盘后
暂停执行；调用方拿到 __interrupt__ 后，用 Command(resume=值) 带着用户输入重新进入，
框架从**同一个检查点**继续跑，已完成节点的副作用不会重放两次（但恢复点的节点会重放，
所以副作用要幂等）。

时间旅行靠的是检查点历史：get_state_history 能按时间倒序列出所有快照，
你可以回到任意一个快照再分叉（fork）出新的执行路径。

常见坑：delete 消息要用 RemoveMessage（add_messages 只会追加）；自定义状态 schema
必须继承 MessagesState，否则 messages 字段没有 reducer，会导致 agent 循环判不出终止条件。
""",
    "rag-architecture.md": """# RAG 架构选型笔记

2-Step RAG：检索永远前置，流程固定、延迟可预测，适合 FAQ 与文档问答。
缺点是"不管用户问什么都要检索一次"，且检索质量差时答案必错。

Agentic RAG：把检索做成工具，由 agent 决定何时检索、检索几次、用什么查询词。
灵活但延迟不定，且可能"该检索时不检索"。

Hybrid：先做一次检索，再让 agent 决定是否继续检索，最后加一道校验环节
（例如用评分器判断答案是否有依据）。

上下文管理的要点：检索回来的全文如果全部塞进主线程，很快就会撑爆窗口。
更稳的做法是把命中内容卸载到文件系统，主线程只保留路径与结论，
细节交给子代理按需读取。
""",
    "ops-runbook.md": """# 值班运行手册

巡检：每两小时记录一次机房温度，超过 28 摄氏度需要在值班群报备并联系设施同事。

发布：发布前必须在预发环境跑一遍全量回归；严禁在未回归的情况下直接上生产。

故障处理：先看监控大盘确认影响面，再查最近一次发布的变更记录。回滚是首选止血手段，
但回滚前要确认没有正在执行的批处理任务，否则可能造成数据不一致。

交接班：把未完成的事项、观察到的异常、以及已采取的临时措施写进交接记录。
""",
}


def build_store() -> InMemoryVectorStore:
    """构建向量库（与 02_langchain/24_RAG知识库 官方补充篇 同一套流程）。"""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=200,
        chunk_overlap=30,
        separators=["\n\n", "\n", "。", "；", "，", " ", ""],
    )
    chunks = splitter.split_documents(
        [Document(page_content=text, metadata={"source": name}) for name, text in DOCS.items()]
    )
    store = InMemoryVectorStore(embedding=embeddings)
    store.add_documents(chunks)
    return store


# ================================================================
# 重排序（SiliconFlow /rerank，与 02_langchain/24 同一实现）
# ================================================================
class SiliconFlowReranker:
    """交叉编码精排：召回后按相关性重排（分数看相对差距，别用绝对阈值）。"""

    def __init__(self) -> None:
        self.url = settings.rerank.base_url.rstrip("/") + "/rerank"
        self.model = settings.rerank.model
        self.api_key = settings.rerank.api_key
        self._opener = urllib.request.build_opener(urllib.request.ProxyHandler({}))

    def rerank(self, query: str, documents: list[str], top_n: int = 3) -> list[int]:
        payload = {
            "model": self.model,
            "query": query,
            "documents": documents,
            "top_n": min(top_n, len(documents)),
        }
        request = urllib.request.Request(
            self.url,
            data=json.dumps(payload).encode("utf-8"),
            headers={"Content-Type": "application/json", "Authorization": f"Bearer {self.api_key}"},
        )
        with self._opener.open(request, timeout=60) as response:
            body = json.loads(response.read().decode("utf-8"))
        return [item["index"] for item in body.get("results", [])]


reranker = SiliconFlowReranker()


def retrieve(query: str, top_k: int = 4, top_n: int = 2) -> list[str]:
    """召回 + 精排，返回最终选中的文本块。"""
    candidates = [document.page_content for document in STORE.similarity_search(query, k=top_k)]
    order = reranker.rerank(query, candidates, top_n=top_n)
    return [candidates[index] for index in order]

### 3.2 Demo 1：知识库构建 + 两级检索（召回 → 精排）

这一步构建内存向量库，并走一遍「召回（embedding 相似度）→ 精排（rerank）」。
打印出来的「总长度 N 字符」是本节的锚点：Demo 2 会把它**整段塞进主线程**，
Demo 3 只把**路径**塞进去 —— 差距就在这儿。

In [ ]:
STORE = build_store()


def demo_1_knowledge_base() -> None:
    print("=" * 70)
    print("Demo 1：知识库构建 + 两级检索（召回 → 精排）")
    print("=" * 70)

    chunks = STORE.store  # InMemoryVectorStore 内部就是 dict：id -> (Document, 向量)
    print(f"  文档 {len(DOCS)} 篇，切片 {len(chunks)} 块（chunk_size=200）")
    query = "检索回来的内容太多怎么办？"
    picked = retrieve(query)
    print(f"  查询：{query}")
    print(f"  两级检索后选中 {len(picked)} 块，总长度 {sum(len(c) for c in picked)} 字符")
    for index, text in enumerate(picked, start=1):
        print(f"    {index}. {text[:56].replace(chr(10), ' ')}…")
    print(
        "  ↑ 记住这个字符数：**Demo 2 会把它整段塞进主线程上下文**，\n"
        "    而 Demo 3 只把路径塞进去 —— 差距就在这儿。"
    )


demo_1_knowledge_base()

### 预期输出

%%%OUTPUT:10%%%

### 3.3 Demo 2：2-Step RAG —— 检索前置，全文进主线程上下文

2-Step RAG 的优点：**流程固定、延迟可预测、控制力强**（每次必检索）。
代价：检索结果**不分青红皂白全部进上下文**，文档一多就撑爆窗口；
而且它没法「追问式检索」（发现资料不够时再去查一次）。

这一格直接 `llm.invoke(prompt)`（把 `prompt` 长度打印出来，供与 Demo 3 对照）。

In [ ]:
def demo_2_two_step_rag() -> None:
    print("\n" + "=" * 70)
    print("Demo 2：2-Step RAG —— 检索前置，全文进主线程上下文")
    print("=" * 70)

    query = "线上接口变慢时应该怎么排查？上下文塞不下怎么办？"
    picked = retrieve(query, top_n=3)
    context = "\n\n".join(f"【资料 {i+1}】{text}" for i, text in enumerate(picked))
    prompt = (
        "你是运维助手。只依据【资料】回答，没有依据就说不知道。"
        "**回答控制在三句话以内**（本机网关对长输出不稳，短答也更适合演示）。\n\n"
        f"{context}\n\n问题：{query}"
    )
    try:
        response = llm.invoke(prompt)
    except Exception as exc:  # noqa: BLE001
        # 网络/网关抖动兜底：不让一次超时把整个演示带崩（按仓库惯例给中文提示）
        print(f"  本次模型调用失败（网关抖动/超时，非代码问题）：{type(exc).__name__}")
        print("  提示：本文件依赖真实模型；网关繁忙时重跑一次即可，其余输出仍可参考。")
        return
    usage = getattr(response, "usage_metadata", None) or {}
    print(f"  送进模型的上下文：{len(prompt)} 字符（{len(picked)} 块全文）")
    print(f"  回答：{str(response.content)[:150]}")
    print(f"  本次 token：输入 {usage.get('input_tokens')} / 输出 {usage.get('output_tokens')}")
    print(
        "  ↑ 2-Step 的优点：**流程固定、延迟可预测、控制力强**（每次必检索）。\n"
        "    代价：检索结果**不分青红皂白全部进上下文**，文档一多就撑爆窗口；\n"
        "    而且它没法「追问式检索」（发现资料不够时再去查一次）。"
    )


demo_2_two_step_rag()

### 预期输出

%%%OUTPUT:11%%%

### 3.4 Demo 3：检索-卸载-委派 —— 全文落盘，主线程只拿路径

官方 `rag.mdx` 的第四个模式，步骤：

```text
① 检索工具把命中块写进文件系统（/retrieved/<id>.md）；
② 工具只返回「路径 + 极短预览」给主 agent；
③ 主 agent 用 task 把「读文件并总结」派给子代理；
④ 子代理在自己的上下文里读全文、总结，只把结论回传主线程。
```

结果：主线程上下文只承载「路径 + 结论」，全文留在文件里。
注意 `SubAgent.model` 是**可选**的：不传就继承主 agent 的模型（`create_deep_agent` 会回填）。
委派不是必然发生：模型有时自己顺手读完就答了 —— 所以在系统提示里**明确要求**委派。

In [ ]:
def demo_3_offload_and_delegate(workdir: Path) -> None:
    print("\n" + "=" * 70)
    print("Demo 3：检索-卸载-委派 —— 全文落盘，主线程只拿路径")
    print("=" * 70)

    # 后端指向临时目录：virtual_mode=True 下 agent 看到的是 "/retrieved/xxx.md" 这样的虚拟路径
    backend = FilesystemBackend(root_dir=workdir, virtual_mode=True)

    @tool
    def search_and_offload(query: str) -> str:
        """检索知识库，把命中的内容写入 /retrieved 目录，返回文件路径清单。"""
        picked = retrieve(query, top_n=2)
        saved: list[str] = []
        for index, text in enumerate(picked, start=1):
            path = f"/retrieved/chunk-{index}.md"
            write_result = backend.write(path, text)
            # 写入失败（路径/权限/编码）要如实回报 —— 否则会把不存在的路径交给子代理，
            # 子代理再 read_file 失败，白烧一轮模型调用。
            if getattr(write_result, "error", None):
                return f"写入 {path} 失败：{write_result.error}"
            saved.append(path)
        # 只回路径 + 极短预览：全文**不进**主线程上下文
        preview = "\n".join(f"{path}（{len(text)} 字符，开头：{text[:24]}…）"
                            for path, text in zip(saved, picked))
        return f"已把 {len(saved)} 段资料写入文件系统：\n{preview}"

    agent = create_deep_agent(
        model=llm,
        backend=backend,
        tools=[search_and_offload],
        subagents=[
            SubAgent(
                name="doc-reader",
                description="读取文件并总结要点（适合处理长文档，不要在主线里读全文）",
                system_prompt="你会收到一个文件路径。读取它，用三句话总结要点，直接给结论。",
                model=llm,
                tools=[],
            )
        ],
        system_prompt=(
            "你是运维助手，工作方式固定：\n"
            "1. 先调用 search_and_offload 检索资料（它会把全文写进文件系统）；\n"
            "2. 然后**必须用 task 工具把「读文件并总结」派给 doc-reader 子代理**，"
            "不要自己在主线里读全文；\n"
            "3. 最后基于子代理的总结回答，两句话以内。"
        ),
    )

    query = "线上接口变慢时应该怎么排查？上下文塞不下怎么办？"
    try:
        result = agent.invoke(
            {"messages": [{"role": "user", "content": query}]},
            config={"recursion_limit": 30},
        )
    except Exception as exc:  # noqa: BLE001
        print(f"  本次 agent 运行失败（网关抖动/超时，非代码问题）：{type(exc).__name__}")
        print("  提示：深度智能体的系统提示很长、每轮推理量也大，网关繁忙时容易超时；")
        print("        本文件已把超时放宽到 300 秒，重跑一次通常即可。")
        return

    tool_sequence = [
        call["name"]
        for message in result["messages"]
        for call in (getattr(message, "tool_calls", None) or [])
    ]
    print(f"  提问：{query}")
    print(f"  工具调用序列：{tool_sequence}")

    # 量化对比：主线程消息总长度（应为路径 + 结论级别，而不是全文级别）
    main_chars = sum(len(str(m.content)) for m in result["messages"])
    full_text_chars = sum(len(text) for text in retrieve(query, top_n=2))
    print(f"\n  主线程消息总长度：{main_chars} 字符")
    print(f"  其中检索到的全文长度：{full_text_chars} 字符（**没有进主线程**，落在文件里）")
    saved_files = sorted(p.name for p in (workdir / "retrieved").glob("*.md")) if (workdir / "retrieved").exists() else []
    print(f"  文件系统里的资料：{saved_files}（可被 read_file / grep 按需读取）")
    print(f"  最终回答：{str(result['messages'][-1].content)[:160]}")
    if "task" in tool_sequence:
        print("  ✔ 本次确实发生了委派（子代理读完文件只回传结论）")
    else:
        print("  （本次主 agent 没走 task —— 委派与否取决于模型；系统提示已强制要求）")
    print(
        "  ↑ 与 Demo 2 对比：\n"
        "    · 2-Step：全文进主线程（上下文 = 全文 + 结论）；\n"
        "    · 卸载-委派：全文进文件系统（上下文 = 路径 + 结论），\n"
        "      细节由子代理在**独立上下文**里消化。\n"
        "    文档规模越大，这个差别的收益越明显（也是课案 03_deepagents/14 讲的\n"
        "    上下文压缩机制的同一个思路：把大块内容挪出上下文）。"
    )


with tempfile.TemporaryDirectory(prefix="delegated_rag_") as tmp:
    demo_3_offload_and_delegate(Path(tmp))
print("\n全部 Demo 执行完毕（临时目录已清理）。")

### 预期输出

%%%OUTPUT:12%%%

## 小结

- **上下文治理（第 1 节，离线）**：deepagents 自带上下文压缩 —— 超长工具结果会被
  **卸载**到文件系统（历史只留「路径 + 预览」，完整内容在 `result["files"]`）；
  `write_todos` 自 v0.7 起 **opt-in**（要显式传 `TodoListMiddleware`）；
  `FilesystemPermission` 三档（`allow`/`deny`/`interrupt`）**先匹配先生效**，
  `interrupt` 把权限和人工审核缝在一起（`Command(resume={"decisions": [...]})`）。
- **Rubric 评分循环（第 2 节，需模型）**：`RubricMiddleware` 是 LLM-as-a-judge 的运行时版本，
  评分器是**独立子代理**，rubric 通过调用时 state 传入、必须配 checkpointer；
  终局判定：`satisfied` / `needs_revision` / `failed` / `grader_error` / `max_iterations_reached`；
  不传 rubric 时是 **no-op**，可无条件常驻。
- **检索-卸载-委派（第 3 节，需模型 + embedding + rerank）**：2-Step RAG 把全文塞进主线程，
  卸载-委派把全文落盘、主线程只拿「路径 + 结论」，子代理在独立上下文里读全文。

**与本机环境的关键结论**：评分器依赖强制 `tool_choice`，思考模型端点（`deepseek-flash`）跑不了，
会落到 `grader_error`（`Thinking mode does not support this tool_choice`），
但脚本仍正常退出 —— **换端点后先看 `on_evaluation` 判定是不是 `grader_error`，别误判成代码问题。**

## 常见坑

1. **`write_todos` 升级后「不会做计划了」**：多半是漏传 `TodoListMiddleware`；
   而且它来自 **langchain** 包（`from langchain.agents.middleware import TodoListMiddleware`），
   不是 deepagents 自己导出的。
2. **权限规则先匹配先生效，一条不匹配就默认放行**：写规则时把最特殊的放最前面；
   `paths` 要锚定（`"/secrets/**"`），`"/**/secrets"` 这类未锚定模式会让批量工具
   （ls/glob/grep）保守地过度触发。
3. **`mode="interrupt"` 必须配 checkpointer**：恢复格式与工具中断一致
   （`Command(resume={"decisions": [...]})`）；忘了 checkpointer 会在中断时报错。
4. **评分循环是双倍调用**（每轮 = 主 agent 一次 + 评分器一次）：用便宜模型当评分器、
   把 `max_iterations` 设小，是最实际的成本控制手段。
5. **rubric 要写成可判定的条目**（有硬指标：要点齐全、字数上限、测试通过）；
   写成「写得好一点」这种主观描述，评分器只能给 failed 或含糊判定。
6. **非 satisfied 结束时消息不被改写**：业务要根据结局分支，必须读回调/事件，
   不要拿最后一条消息去猜（官方 note 专门提醒过）。
7. **卸载路径要固定且可枚举**（本课用 `/retrieved/chunk-N.md`），且只回传「路径 + 极短预览」；
   一旦把全文也塞进返回值，这个模式就退回成普通 RAG 了（白折腾）。
8. **`SubAgent.model` 是可选**的：不传就继承主 agent 的模型；但子代理的 `description`
   要写清「适合读长文档」，否则主 agent 不会把活派给它。委派不是必然发生，要在系统提示里明确要求。
9. **该中间件是 beta**，升级 deepagents 后请优先回归第 2 节。

## 官方链接

- 上下文工程（卸载 / 摘要 / compact）：<https://docs.langchain.com/oss/python/deepagents/context-engineering>
- 文件权限（FilesystemPermission）：<https://docs.langchain.com/oss/python/deepagents/permissions>
- Rubric 评分循环（RubricMiddleware）：<https://docs.langchain.com/oss/python/deepagents/rubric>
- 检索-卸载-委派（rag / retrieval）：<https://docs.langchain.com/oss/python/deepagents/retrieval>
- 任务规划（TodoList / write_todos）：<https://docs.langchain.com/oss/python/deepagents/overview>